In [1]:
#Load and check the outlier count matches
import duckdb
import pandas as pd

health = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/health_score.parquet')"
).df()

print("Total listings:", len(health))
print("price_outlier_flag count:", health["flag_price_outlier"].sum(), "(should be 101)")
print()
print(health["health_score"].describe().round(1))

Total listings: 21514
price_outlier_flag count: 101 (should be 101)

count    21514.0
mean        79.3
std         17.2
min          0.0
25%         65.0
50%         80.0
75%         90.0
max        100.0
Name: health_score, dtype: float64


In [2]:
#Look at the worst-scoring listings
flags = ["flag_overpriced", "flag_underpriced", "flag_high_availability",
         "flag_gone_quiet", "flag_rating_gap", "flag_restrictive_stay", "flag_price_outlier"]

worst = health.sort_values("health_score").head(10)
print(worst[["listing_id", "room_type", "stay_type", "health_score"] + flags].to_string(index=False))

         listing_id       room_type    stay_type  health_score  flag_overpriced  flag_underpriced  flag_high_availability  flag_gone_quiet  flag_rating_gap  flag_restrictive_stay  flag_price_outlier
           38521231    Private room   short stay             0            False              True                   False            False            False                   <NA>               False
           49920227    Private room monthly stay             5             True             False                    True             True             True                  False                True
           39553889    Private room   short stay             5             True             False                    True             True             True                  False                True
             990529    Private room monthly stay            20             True             False                    True             True            False                  False                True
     

In [3]:
#How common is each flag, and do any overlap heavily?
print(health[flags].mean().mul(100).round(1))
print()
print("Listings with 0 flags:", (health[flags].sum(axis=1) == 0).sum())
print("Listings with 3+ flags:", (health[flags].sum(axis=1) >= 3).sum())

flag_overpriced           28.4
flag_underpriced          25.7
flag_high_availability    34.2
flag_gone_quiet           26.4
flag_rating_gap            8.2
flag_restrictive_stay      3.5
flag_price_outlier         0.5
dtype: Float64

Listings with 0 flags: 5007
Listings with 3+ flags: 2401


In [4]:
#Spot-check one flag manually
# Pick a "gone quiet" listing and confirm it by hand
sample = health[health["flag_gone_quiet"]].iloc[0]
print(sample[["listing_id", "number_of_reviews", "number_of_reviews_ltm", "flag_gone_quiet"]])

listing_id               263005
number_of_reviews           129
number_of_reviews_ltm         0
flag_gone_quiet            True
Name: 489, dtype: object


In [5]:
# Find all rows where flag_restrictive_stay is null, and inspect their segment
null_flag = health[health["flag_restrictive_stay"].isna()]
print("Rows with null flag_restrictive_stay:", len(null_flag))
print()
print(null_flag[["listing_id", "borough", "room_type", "stay_type",
                  "minimum_nights", "avg_min_nights_short", "health_score"]].head(10))

# And check: does health_score for these rows look wrong (too harsh, since a NULL
# deduction may have propagated) or did GREATEST/COALESCE somehow save it?
print()
print("health_score stats for these null-flag rows:")
print(null_flag["health_score"].describe())

Rows with null flag_restrictive_stay: 1

       listing_id   borough     room_type   stay_type  minimum_nights  \
10359    38521231  Brooklyn  Private room  short stay            <NA>   

       avg_min_nights_short  health_score  
10359              1.930811             0  

health_score stats for these null-flag rows:
count    1.0
mean     0.0
std      NaN
min      0.0
25%      0.0
50%      0.0
75%      0.0
max      0.0
Name: health_score, dtype: float64


In [6]:
raw_check = duckdb.sql("""
    SELECT listing_id, minimum_nights, quote_nights, base_price, price_confidence
    FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')
    WHERE listing_id = 38521231
""").df()
print(raw_check)

# Is this the only listing with a NULL minimum_nights among priced listings?
null_min_nights = duckdb.sql("""
    SELECT COUNT(*) AS count
    FROM read_parquet('../data/gold/nyc/listings_prepared.parquet')
    WHERE base_price IS NOT NULL AND minimum_nights IS NULL
""").df()
print(null_min_nights)

   listing_id  minimum_nights  quote_nights  base_price price_confidence
0    38521231            <NA>             2       126.5           normal
   count
0      1


In [7]:
health = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/health_score.parquet')"
).df()
print("Rows with null health_score:", health["health_score"].isna().sum())
print("Rows with null flag_restrictive_stay:", health["flag_restrictive_stay"].isna().sum())
print(health.loc[health["listing_id"] == 38521231, ["health_score", "flag_restrictive_stay"]])

Rows with null health_score: 0
Rows with null flag_restrictive_stay: 0
       health_score  flag_restrictive_stay
10359            90                  False
